# Phase 3 – Soft-voting Ensemble (Logistic Regression + Random Forest + Gradient Boosting)

**Idea in one sentence:** ask three *different* models for a risk estimate and take the **average**. If they make different mistakes, the average is steadier than any single one.

**Roadmap**
1. Load data (same as Phase 2)
2. Build the three models and the ensemble
3. Compare all of them on **identical CV splits**
4. Test whether the ensemble is **really** better (paired comparison)
5. Diversity check – are the models actually different?
6. Recalibrate the probabilities
7. Out-of-fold results: threshold, calibration, clinician-confidence check
8. Final model, what the trees find important, predictions for unlabelled patients

Every block starts with **Why**.

## 1. Load data

**Why:** exactly the same data and the same 56 features as Phase 2, so any difference we see comes from the *model*, not the data.

In [1]:
import json, warnings
import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, VotingClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.model_selection import RepeatedStratifiedKFold, StratifiedKFold, cross_validate
from sklearn.metrics import roc_auc_score, roc_curve, confusion_matrix
warnings.filterwarnings('ignore')
pd.set_option('display.width', 150)

DATA_DIR = '.'
data = pd.read_csv(f'{DATA_DIR}/features_clean.csv')
meta = json.load(open(f'{DATA_DIR}/feature_lists.json'))

RARE = ['flag_anticoagulant', 'dre_not_done', 'zone_unknown']       # too few positives to learn from
FEATURES = [c for c in meta['features'] if c not in RARE]

is_labelled = data['label'].notna()
X_all, ids_all = data[FEATURES], data['case_id']
X, y, ids = X_all[is_labelled], data.loc[is_labelled, 'label'].astype(int), ids_all[is_labelled]
print(f'labelled: {len(y)} | unlabelled: {(~is_labelled).sum()} | features: {len(FEATURES)}')

labelled: 91 | unlabelled: 104 | features: 56


## 2. The three models and the ensemble

**Why these three?** They learn in different ways, so their errors differ:

| Model | How it thinks | Settings and why |
|---|---|---|
| **Logistic Regression** | one straight-line rule: each feature adds or subtracts risk | `C=0.01` – strong penalty, because Phase 2 showed small `C` works best with 91 patients |
| **Random Forest** | many decision trees on random subsets, then vote; can catch "if A *and* B" patterns | `max_depth=3`, `min_samples_leaf=5` – **shallow trees** so they cannot memorise individual patients |
| **Gradient Boosting** | small trees built one after another, each fixing the previous errors | `max_depth=2`, `learning_rate=0.05`, `subsample=0.7` – deliberately slow and weak, because boosting overfits easily on tiny data |

**Why no heavy tuning?** Tuning many settings on 91 patients mostly fits noise. We use cautious, standard settings instead.

**Why imputer only for trees (and scaler only for LR)?** Missing values must be filled for all three. Scaling matters for LR (its penalty treats features equally) but trees ignore feature scale, so we skip it.

**Soft voting** = average the predicted probabilities (instead of counting yes/no votes). It keeps the models' confidence and gives smooth risk values. Equal weights = no extra settings to overfit.

In [2]:
def lr_model():
    return Pipeline([('impute', SimpleImputer(strategy='median')),
                     ('scale',  StandardScaler()),
                     ('model',  LogisticRegression(C=0.01, max_iter=5000))])

def rf_model():
    return Pipeline([('impute', SimpleImputer(strategy='median')),
                     ('model',  RandomForestClassifier(n_estimators=300, max_depth=3, min_samples_leaf=5,
                                                       max_features='sqrt', random_state=0, n_jobs=-1))])

def gb_model():
    return Pipeline([('impute', SimpleImputer(strategy='median')),
                     ('model',  GradientBoostingClassifier(n_estimators=100, learning_rate=0.05, max_depth=2,
                                                           subsample=0.7, min_samples_leaf=5, random_state=0))])

def ensemble_model():
    return VotingClassifier([('lr', lr_model()), ('rf', rf_model()), ('gb', gb_model())],
                            voting='soft')          # equal weights by default

## 3. Compare everything on the same CV splits

**Why:** repeated stratified 5-fold CV (10 repeats = 50 scores), as in Phase 2. We fix the random seed so **every model gets exactly the same 50 train/test splits**. Then differences come from the models, not from lucky splits.

In [3]:
cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=10, random_state=42)

models = {'Logistic Regression': lr_model(), 'Random Forest': rf_model(),
          'Gradient Boosting': gb_model(), 'Ensemble (average of 3)': ensemble_model()}

scores = {}
for name, model in models.items():
    scores[name] = cross_validate(model, X, y, cv=cv, scoring='roc_auc')['test_score']
    print(f'{name:26s} AUC = {scores[name].mean():.3f}  (± {scores[name].std():.3f})')

Logistic Regression        AUC = 0.714  (± 0.122)


Random Forest              AUC = 0.705  (± 0.110)


Gradient Boosting          AUC = 0.718  (± 0.107)


Ensemble (average of 3)    AUC = 0.732  (± 0.108)


**How to read this:** AUC is the chance that a random biopsy patient gets a higher risk than a random no-biopsy patient (0.5 = coin flip). The `±` shows how much the score moves between splits.

## 4. Is the ensemble *really* better? (paired comparison)

**Why:** the averages above can differ by chance. Because all models used the same 50 splits, we can compare them **split by split**: on how many of the 50 splits does the ensemble beat each single model, and by how much on average? An improvement that wins only about half the splits is not convincing.

In [4]:
ens = scores['Ensemble (average of 3)']
rows = []
for name in ['Logistic Regression', 'Random Forest', 'Gradient Boosting']:
    diff = ens - scores[name]
    rows.append({'ensemble vs': name, 'mean AUC gain': diff.mean(),
                 'ensemble wins in % of splits': 100 * (diff > 0).mean(),
                 'std of gain': diff.std()})
print(pd.DataFrame(rows).round(3).to_string(index=False))

        ensemble vs  mean AUC gain  ensemble wins in % of splits  std of gain
Logistic Regression          0.018                          64.0        0.065
      Random Forest          0.027                          64.0        0.055
  Gradient Boosting          0.014                          56.0        0.048


**How to read this:** a gain of about 0.01–0.02 with a std of about 0.05 is *not* a clear win. Look for a gain that is both positive **and** wins clearly more than 50% of the splits. Report the honest answer either way.

## 5. Diversity check

**Why:** an ensemble only helps if its members **disagree sometimes**. If all three give nearly the same predictions, averaging adds nothing. We get one honest prediction per patient from each model (out-of-fold: the model never saw that patient) and look at how correlated they are.

In [5]:
def out_of_fold(make_model, n_repeats=5):
    # predictions for each patient from models that never saw that patient, averaged over repeats
    pred = np.zeros(len(y))
    for seed in range(n_repeats):
        for tr, te in StratifiedKFold(5, shuffle=True, random_state=seed).split(X, y):
            m = make_model().fit(X.iloc[tr], y.iloc[tr])
            pred[te] += m.predict_proba(X.iloc[te])[:, 1] / n_repeats
    return pred

oof = {'LR': out_of_fold(lr_model), 'RF': out_of_fold(rf_model), 'GB': out_of_fold(gb_model)}
oof['Ensemble'] = np.mean([oof['LR'], oof['RF'], oof['GB']], axis=0)

print('Correlation between model predictions (1 = identical, lower = more diverse):')
print(pd.DataFrame(oof).drop(columns='Ensemble').corr().round(2))
print('\nOut-of-fold AUC:', {k: round(roc_auc_score(y, v), 3) for k, v in oof.items()})

Correlation between model predictions (1 = identical, lower = more diverse):
      LR    RF    GB
LR  1.00  0.84  0.68
RF  0.84  1.00  0.84
GB  0.68  0.84  1.00

Out-of-fold AUC: {'LR': 0.718, 'RF': 0.698, 'GB': 0.753, 'Ensemble': 0.759}


**How to read this:** correlations around 0.9+ mean the models are nearly copies; around 0.3–0.7 means useful diversity. Low correlation also means the models may be partly fitting noise differently, so a diverse ensemble is not automatically a good one.

## 6. Does recalibrating the probabilities help? (test it, don't assume)

**Why look at this:** in Phase 2 the predicted risks were squeezed into a narrow range. **Platt scaling** (`method='sigmoid'`, only 2 learned numbers) can stretch them to match reality. It wraps the ensemble and is learned **inside each CV fold** from training patients only, so there is no leakage. We use the sigmoid rather than isotonic because 91 patients is too few for the flexible version.

**The catch:** calibration needs extra data to learn from, and with 91 patients that can cost more than it gives. So we measure it with two scores and keep it **only if it improves them**:
- **AUC**: ranking quality (calibration should not change it much)
- **Brier score**: how close probabilities are to what really happened (lower = better; always guessing the base rate gives about 0.24)

In [6]:
def calibrated_ensemble():
    return CalibratedClassifierCV(ensemble_model(), method='sigmoid', cv=5)

cal_scores = cross_validate(calibrated_ensemble(), X, y, cv=cv, scoring=['roc_auc', 'neg_brier_score'])
raw_scores = cross_validate(ensemble_model(),      X, y, cv=cv, scoring=['roc_auc', 'neg_brier_score'])

print(f"Ensemble              AUC = {raw_scores['test_roc_auc'].mean():.3f}   Brier = {-raw_scores['test_neg_brier_score'].mean():.3f}")
print(f"Ensemble + calibration AUC = {cal_scores['test_roc_auc'].mean():.3f}   Brier = {-cal_scores['test_neg_brier_score'].mean():.3f}")

# Keep calibration only if it lowers the Brier score (i.e. gives better probabilities)
use_calibration = (-cal_scores['test_neg_brier_score'].mean()) < (-raw_scores['test_neg_brier_score'].mean())
final_factory = calibrated_ensemble if use_calibration else ensemble_model
print('\nUsing calibration in the final model:', use_calibration)

Ensemble              AUC = 0.732   Brier = 0.206
Ensemble + calibration AUC = 0.713   Brier = 0.213

Using calibration in the final model: False


**How to read this:** if calibration does *not* lower the Brier score, the ensemble's probabilities were already reasonable (averaging Random Forest and Boosting spreads them more than Logistic Regression alone), and the extra step only adds noise. Then we skip it.

## 7. Out-of-fold results for the final ensemble

**Why:** we now look at honest per-patient predictions from the final ensemble, just like Phase 2, so the two phases can be compared directly.

In [7]:
oof_final = out_of_fold(final_factory)
print('AUC of out-of-fold predictions: %.3f' % roc_auc_score(y, oof_final))
print('range of predicted risk: %.2f to %.2f' % (oof_final.min(), oof_final.max()))

AUC of out-of-fold predictions: 0.759
range of predicted risk: 0.38 to 0.86


### 7a. Threshold
**Why:** missing a patient who needs a biopsy is worse than an unnecessary biopsy, so we pick the threshold that keeps **sensitivity at 90% or higher** and see what specificity remains. (The threshold is picked on the same predictions we report, so treat it as an estimate.)

In [8]:
fpr, tpr, thr = roc_curve(y, oof_final)
threshold = thr[np.where(tpr >= 0.90)[0][0]]
pred = (oof_final >= threshold).astype(int)
tn, fp, fn, tp = confusion_matrix(y, pred).ravel()
print(f'threshold = {threshold:.2f}')
print(f'sensitivity: {tp/(tp+fn):.2f}   specificity: {tn/(tn+fp):.2f}')
print(pd.DataFrame([[tn, fp], [fn, tp]], index=['truth: no biopsy', 'truth: biopsy'],
                   columns=['pred: no biopsy', 'pred: biopsy']))

threshold = 0.49
sensitivity: 0.91   specificity: 0.31
                  pred: no biopsy  pred: biopsy
truth: no biopsy               11            24
truth: biopsy                   5            51


### 7b. Calibration table
**Why:** if the model says 70%, about 70 of 100 such patients should truly need a biopsy. We compare predicted vs actual rates in 4 risk bands. Compare with Phase 2, where *predicted* stayed between 0.56 and 0.67.

In [9]:
bands = pd.qcut(oof_final, 4, labels=['lowest', 'low-mid', 'high-mid', 'highest'])
print(pd.DataFrame({'predicted': oof_final, 'actual': y.values}).groupby(bands).mean().round(2)
        .assign(n=pd.Series(bands).value_counts()))

          predicted  actual   n
lowest         0.45    0.39  23
low-mid        0.59    0.39  23
high-mid       0.68    0.82  22
highest        0.78    0.87  23


### 7c. Clinician-confidence check
**Why:** `reasoning_audit.csv` is written after the decision, so it is **never** used as an input. We only use it to ask: are the model's mistakes concentrated in the cases the clinician found uncertain?

In [10]:
audit = pd.read_csv(f'{DATA_DIR}/reasoning_audit.csv')[['case_id', 'confidence']]
chk = pd.DataFrame({'case_id': ids.values, 'y': y.values, 'p': oof_final}).merge(audit, on='case_id', how='left')
chk['abs_error'] = (chk['y'] - chk['p']).abs()
print(chk.groupby('confidence').agg(patients=('y', 'size'), mean_abs_error=('abs_error', 'mean')).round(3))

            patients  mean_abs_error
confidence                          
borderline        18           0.439
clear             58           0.400
uncertain         15           0.458


## 8. Final model, important features, and predictions for unlabelled patients

**Why:** after trusting the evaluation, we fit the final ensemble on **all 91 labelled patients** (more data = better model) and use it on the 104 unlabelled ones. Those patients have no ground truth, so we cannot score these predictions.

**Feature importance from the Random Forest:** a rough look at which features the trees use most. It is only a guide (importance is shared between correlated features, and it is not proof of cause).

In [11]:
final_model = final_factory().fit(X, y)

rf_only = rf_model().fit(X, y)
imp = pd.Series(rf_only.named_steps['model'].feature_importances_, index=FEATURES).sort_values(ascending=False)
print('Top 10 features by Random Forest importance:'); print(imp.head(10).round(3))

risk = final_model.predict_proba(X_all[~is_labelled])[:, 1]
out = pd.DataFrame({'case_id': ids_all[~is_labelled].values,
                    'biopsy_risk': risk.round(3),
                    'recommend_biopsy': (risk >= threshold).astype(int)})
out.to_csv('phase3_ensemble_unlabelled_predictions.csv', index=False)
print('\n', out['recommend_biopsy'].value_counts().rename({1: 'biopsy', 0: 'no biopsy'}).to_string())
out.head()

Top 10 features by Random Forest importance:
pirads                                          0.067
vol_log                                         0.050
psa_slope_per_step_log                          0.050
lab_ldh                                         0.043
enc_type_Re-evaluation - prior PCa diagnosis    0.039
hr_bpm                                          0.039
cspca                                           0.038
lab_haemoglobin                                 0.036
prior_isup_grade                                0.035
psa_last_over_first_log                         0.034
dtype: float64

 recommend_biopsy
biopsy       78
no biopsy    26


,case_id,biopsy_risk,recommend_biopsy
0,PT-pseudo_598f1d8f4e08,0.631,1
1,PT-pseudo_5af00f9153c2,0.616,1
2,PT-pseudo_5dc2eb70fbd7,0.452,0
3,PT-pseudo_5f263f325019,0.670,1
4,PT-pseudo_62c93b0c3dba,0.674,1


## Summary checklist (what to report)
- AUC of each single model and of the ensemble on identical splits (section 3)
- The paired comparison: the gain and the % of splits the ensemble wins (section 4)
- The diversity check (section 5)
- Whether calibration helped (Brier score, section 6) and the calibration table (7b)
- Threshold, sensitivity and specificity (7a), and the clinician-confidence check (7c)
- Honest caveat: with 91 labelled patients, differences smaller than the `±` spread are not reliable